# Mission 06 — Autonomous Warp State Machine & Deliberation Loop
**Squadron LeCun** · Sector SEC-06-CAR (Eta Carinae & Homunculus Nebula) · Subsystem 06

| Part | Role | Section |
|---|---|---|
| 1 | Systems Architect | Agent State Machine Core + step budget |
| 2 | Communications & Protocol Officer | Tool protocol, JSON-Schema validator, error boundary |
| 3 | Data & Payload Specialist | MDP trajectory logger, discounted return, transition matrix |
| 4 | Security & Ethics Sentinel | Floyd cycle detection, state-action hashing, emergency tripwire |

Pure Python standard library only (Enum, dataclasses, json, hashlib). Run all cells top to bottom.

## Part 1 — Systems Architect: Agent State Machine Core

In [1]:
"""
Mission 06 — Part 1: Systems Architect (Lead Engineer)
Native Python Agent State Machine Core & Step Budget Enforcement

Sector SEC-06-CAR (Eta Carinae & Homunculus Nebula)
Subsystem 06: Autonomous Warp State Machine & Deliberation Loop

Pure standard library: Enum + dataclass. No agent frameworks.

Transition graph (exactly one transition per step() call):
    IDLE -> PERCEIVING -> DELIBERATING -> ACTING -> OBSERVING -> REFLECTING
    REFLECTING -> DELIBERATING (loop)  |  HALTED (terminal)
Step budget: max_steps N <= 20. Once N transitions are used, the next
step() forcibly moves the agent to HALTED.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, Dict, FrozenSet, List, Optional

MAX_STEP_BUDGET = 20  # hard ceiling from the mission brief (N <= 20)


class AgentState(Enum):
    IDLE = "IDLE"
    PERCEIVING = "PERCEIVING"
    DELIBERATING = "DELIBERATING"
    ACTING = "ACTING"
    OBSERVING = "OBSERVING"
    REFLECTING = "REFLECTING"
    HALTED = "HALTED"


# The strict, legal transition graph. step() may only follow these edges
# (plus the forced budget edge "any state -> HALTED").
TRANSITIONS: Dict[AgentState, FrozenSet[AgentState]] = {
    AgentState.IDLE: frozenset({AgentState.PERCEIVING}),
    AgentState.PERCEIVING: frozenset({AgentState.DELIBERATING}),
    AgentState.DELIBERATING: frozenset({AgentState.ACTING}),
    AgentState.ACTING: frozenset({AgentState.OBSERVING}),
    AgentState.OBSERVING: frozenset({AgentState.REFLECTING}),
    AgentState.REFLECTING: frozenset({AgentState.DELIBERATING, AgentState.HALTED}),
    AgentState.HALTED: frozenset(),  # terminal: no way out
}


class IllegalTransitionError(RuntimeError):
    """Raised if code ever tries to move along an edge not in TRANSITIONS."""


@dataclass
class Agent:
    """AURA's deterministic execution core.

    `goal_reached` is decided in REFLECTING: True -> HALTED, False -> loop back
    to DELIBERATING. Optional hooks let the other subsystems (tool dispatcher,
    trajectory logger, loop sentinel) plug into each state without changing
    the state machine itself.
    """

    name: str = "AURA"
    max_steps: int = MAX_STEP_BUDGET
    state: AgentState = AgentState.IDLE
    step_count: int = 0
    goal_reached: bool = False
    halt_reason: Optional[str] = None
    context: List[Dict[str, Any]] = field(default_factory=list)   # observations, errors, notes
    history: List[tuple] = field(default_factory=list)            # (step, from, to)
    hooks: Dict[AgentState, Callable[["Agent"], None]] = field(default_factory=dict)
    verbose: bool = False

    def __post_init__(self) -> None:
        if not isinstance(self.max_steps, int) or not (1 <= self.max_steps <= MAX_STEP_BUDGET):
            raise ValueError(
                f"max_steps must be an int in [1, {MAX_STEP_BUDGET}], got {self.max_steps!r}"
            )

    # ---------- core ----------
    def _next_state(self) -> AgentState:
        if self.state is AgentState.REFLECTING:
            return AgentState.HALTED if self.goal_reached else AgentState.DELIBERATING
        (only_next,) = TRANSITIONS[self.state]  # every other state has exactly one successor
        return only_next

    def _move(self, new_state: AgentState, forced: bool = False) -> None:
        if not forced and new_state not in TRANSITIONS[self.state]:
            raise IllegalTransitionError(f"{self.state.name} -> {new_state.name} is not allowed")
        self.history.append((self.step_count, self.state, new_state))
        if self.verbose:
            tag = " [FORCED]" if forced else ""
            print(f"step {self.step_count:02d}: {self.state.name:<12} -> {new_state.name}{tag}")
        self.state = new_state

    def halt(self, reason: str) -> None:
        """Force HALTED from any state (used by the budget and by the Sentinel's tripwire)."""
        if self.state is AgentState.HALTED:
            return
        self.halt_reason = reason
        self._move(AgentState.HALTED, forced=True)

    def step(self) -> AgentState:
        """Execute exactly ONE state transition and return the new state."""
        if self.state is AgentState.HALTED:
            return self.state  # terminal: further calls are no-ops

        if self.step_count >= self.max_steps:
            self.halt(f"step budget exhausted ({self.step_count}/{self.max_steps})")
            return self.state

        self.step_count += 1
        new_state = self._next_state()
        if new_state is AgentState.HALTED:
            self.halt_reason = "goal reached"
        self._move(new_state)

        hook = self.hooks.get(new_state)
        if hook is not None and self.state is new_state:
            hook(self)
        return self.state

    def run(self, max_calls: int = 1000) -> AgentState:
        """Call step() until HALTED (max_calls is just a safety net for the caller)."""
        for _ in range(max_calls):
            if self.state is AgentState.HALTED:
                break
            self.step()
        return self.state


## Part 2 — Communications & Protocol Officer: Tool Protocol & Action Dispatcher

In [2]:
"""
Mission 06 — Part 2: Communications & Protocol Officer
Tool Protocol, JSON-Schema Validator & Action Dispatcher with Error Boundary

Sector SEC-06-CAR (Eta Carinae & Homunculus Nebula)
Subsystem 06: Autonomous Warp State Machine & Deliberation Loop

Flow for every tool request from the LLM:
    raw JSON text -> parse -> validate against the tool's schema -> execute
Any failure at any stage becomes an observation string
    "Tool Execution Error: <msg>"
which is injected into the agent's context so the next deliberation cycle can
fix the mistake. Nothing ever propagates as a Python exception.
"""

from __future__ import annotations

import json
from typing import Any, Callable, Dict, List, Optional, Union


# ---------------------------------------------------------------------------
# 1. Tools
# ---------------------------------------------------------------------------
def navigate_warp_vector(x: float, y: float, z: float) -> str:
    """Set the warp course vector through the nebula."""
    return f"Warp vector aligned to [{x:.2f}, {y:.2f}, {z:.2f}]."


def adjust_plasma_shields(intensity: int) -> str:
    """Set plasma shield intensity in percent (0..100)."""
    if not 0 <= intensity <= 100:
        raise ValueError(f"Shield intensity {intensity}% out of range [0, 100].")
    return f"Plasma shields set to {intensity}%."


def query_sensor_array(target: str) -> str:
    """Scan a target in the nebula."""
    readings = {
        "homunculus_core": "Severe plasma flare. Dense stellar dust at azimuth 142.",
        "eta_carinae_b": "O-type companion emitting ~100x solar luminosity.",
        "escape_corridor": "Clear gravitational corridor along vector [-0.85, 0.52, 0.05].",
    }
    key = target.strip().lower()
    if not key:
        raise ValueError("Sensor target must not be empty.")
    return readings.get(key, f"Target '{target}' unresolved.")


# ---------------------------------------------------------------------------
# 2. Tool contracts (JSON-Schema subset) + a strict validator (no external libs)
# ---------------------------------------------------------------------------
TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "navigate_warp_vector": {
        "type": "object",
        "properties": {"x": {"type": "number"}, "y": {"type": "number"}, "z": {"type": "number"}},
        "required": ["x", "y", "z"],
        "additionalProperties": False,
    },
    "adjust_plasma_shields": {
        "type": "object",
        # Range check is left to the tool on purpose, so the error boundary
        # can be demonstrated with the ValueError from the brief.
        "properties": {"intensity": {"type": "integer"}},
        "required": ["intensity"],
        "additionalProperties": False,
    },
    "query_sensor_array": {
        "type": "object",
        "properties": {"target": {"type": "string", "minLength": 1}},
        "required": ["target"],
        "additionalProperties": False,
    },
}

TOOL_REGISTRY: Dict[str, Callable[..., str]] = {
    "navigate_warp_vector": navigate_warp_vector,
    "adjust_plasma_shields": adjust_plasma_shields,
    "query_sensor_array": query_sensor_array,
}


def _type_ok(value: Any, expected: str) -> bool:
    # bool is a subclass of int in Python, so it is excluded explicitly.
    if expected == "number":
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    if expected == "integer":
        return isinstance(value, int) and not isinstance(value, bool)
    if expected == "string":
        return isinstance(value, str)
    if expected == "boolean":
        return isinstance(value, bool)
    if expected == "object":
        return isinstance(value, dict)
    if expected == "array":
        return isinstance(value, list)
    return False


def validate_against_schema(args: Any, schema: Dict[str, Any]) -> List[str]:
    """Return a list of violations (empty list = valid)."""
    errors: List[str] = []
    if not _type_ok(args, schema.get("type", "object")):
        return [f"arguments must be a JSON {schema.get('type', 'object')}, got {type(args).__name__}"]

    props = schema.get("properties", {})
    for req in schema.get("required", []):
        if req not in args:
            errors.append(f"missing required argument '{req}'")
    if schema.get("additionalProperties", True) is False:
        for key in args:
            if key not in props:
                errors.append(f"unexpected argument '{key}'")
    for key, rule in props.items():
        if key not in args:
            continue
        val = args[key]
        if not _type_ok(val, rule["type"]):
            errors.append(f"'{key}' must be {rule['type']}, got {type(val).__name__} ({val!r})")
            continue
        if rule["type"] in ("number", "integer") and val != val:  # NaN
            errors.append(f"'{key}' must not be NaN")
        if "minimum" in rule and val < rule["minimum"]:
            errors.append(f"'{key}' must be >= {rule['minimum']}")
        if "maximum" in rule and val > rule["maximum"]:
            errors.append(f"'{key}' must be <= {rule['maximum']}")
        if "minLength" in rule and len(val) < rule["minLength"]:
            errors.append(f"'{key}' must have length >= {rule['minLength']}")
    return errors


# ---------------------------------------------------------------------------
# 3. Action dispatcher with error boundary
# ---------------------------------------------------------------------------
ERROR_PREFIX = "Tool Execution Error: "


class ActionDispatcher:
    """Parse -> validate -> execute. Always returns a string observation."""

    def __init__(self, registry=None, schemas=None):
        self.registry = registry if registry is not None else TOOL_REGISTRY
        self.schemas = schemas if schemas is not None else TOOL_SCHEMAS
        self.log: List[Dict[str, Any]] = []

    def _error(self, msg: str) -> str:
        return ERROR_PREFIX + msg

    def dispatch(self, request: Union[str, Dict[str, Any]], arguments: Optional[Any] = None) -> str:
        """
        Accepts either
          dispatch('{"tool": "...", "arguments": {...}}')        # raw LLM JSON
          dispatch({"tool": "...", "arguments": {...}})
          dispatch("tool_name", {...})
        """
        try:
            tool, args = self._parse(request, arguments)
            if isinstance(tool, str) and tool.startswith(ERROR_PREFIX):
                obs = tool
            elif tool not in self.registry:
                obs = self._error(f"unknown tool '{tool}'. Available: {sorted(self.registry)}")
            else:
                problems = validate_against_schema(args, self.schemas[tool])
                if problems:
                    obs = self._error(f"invalid arguments for '{tool}': " + "; ".join(problems))
                else:
                    obs = "Observation: " + self.registry[tool](**args)
        except Exception as exc:  # the error boundary: nothing escapes
            obs = self._error(f"{type(exc).__name__}: {exc}")
        self.log.append({"request": request if arguments is None else (request, arguments), "observation": obs})
        return obs

    def _parse(self, request, arguments):
        if arguments is not None:  # dispatch("tool", {...})
            return request, arguments
        if isinstance(request, str):
            try:
                request = json.loads(request)
            except json.JSONDecodeError as exc:
                return self._error(f"malformed JSON payload ({exc.msg} at pos {exc.pos})"), None
        if not isinstance(request, dict):
            return self._error("payload must be a JSON object with 'tool' and 'arguments'"), None
        if "tool" not in request:
            return self._error("payload missing 'tool' field"), None
        return request["tool"], request.get("arguments", {})

    def dispatch_into(self, agent, request, arguments=None) -> str:
        """Dispatch and inject the observation (or error) into agent.context."""
        obs = self.dispatch(request, arguments)
        agent.context.append({
            "role": "observation",
            "is_error": obs.startswith(ERROR_PREFIX),
            "content": obs,
        })
        return obs


## Part 3 — Data & Payload Specialist: MDP Trajectory Logger & Return Evaluator

In [3]:
"""
Mission 06 — Part 3: Data & Payload Specialist
MDP Trajectory Logger & Empirical Return Evaluator

Sector SEC-06-CAR (Eta Carinae & Homunculus Nebula)
Subsystem 06: Autonomous Warp State Machine & Deliberation Loop

1. Trajectory logging   tau = (s0, a0, r0, s1, a1, r1, ..., sT)
2. Discounted return    G0 = sum_t gamma^t * r_t        (gamma = 0.95)
3. Transition matrix    P(s' | s, a) = count(s, a, s') / count(s, a)
                        estimated from a historical dataset of past jumps.
"""

from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, Hashable, Iterable, List, Optional, Sequence, Tuple

GAMMA = 0.95


# ---------------------------------------------------------------------------
# 1. Trajectory logger
# ---------------------------------------------------------------------------
@dataclass
class Transition:
    t: int
    state: Hashable
    action: Any
    reward: float
    next_state: Hashable


@dataclass
class TrajectoryLogger:
    """Records one episode. Call start(s0), then log(a, r, s_next) each step."""

    states: List[Hashable] = field(default_factory=list)
    actions: List[Any] = field(default_factory=list)
    rewards: List[float] = field(default_factory=list)
    done: bool = False

    def start(self, initial_state: Hashable) -> None:
        self.states, self.actions, self.rewards, self.done = [initial_state], [], [], False

    def log(self, action: Any, reward: float, next_state: Hashable, done: bool = False) -> None:
        if not self.states:
            raise RuntimeError("call start(s0) before log()")
        if self.done:
            raise RuntimeError("episode already finished")
        self.actions.append(action)
        self.rewards.append(float(reward))
        self.states.append(next_state)
        self.done = done

    @property
    def T(self) -> int:
        return len(self.actions)

    def transitions(self) -> List[Transition]:
        return [Transition(t, self.states[t], self.actions[t], self.rewards[t], self.states[t + 1])
                for t in range(self.T)]

    def as_tau(self) -> Tuple:
        """Flat tuple (s0, a0, r0, s1, a1, r1, ..., sT)."""
        out: List[Any] = []
        for t in range(self.T):
            out += [self.states[t], self.actions[t], self.rewards[t]]
        out.append(self.states[-1])
        return tuple(out)

    def discounted_return(self, gamma: float = GAMMA) -> float:
        return discounted_return(self.rewards, gamma)

    def returns_to_go(self, gamma: float = GAMMA) -> List[float]:
        """G_t for every t (useful to see long-term vs short-term value)."""
        g, out = 0.0, []
        for r in reversed(self.rewards):
            g = r + gamma * g
            out.append(g)
        return out[::-1]

    def pretty(self) -> str:
        lines = [f"s0 = {self.states[0]}"]
        for tr in self.transitions():
            lines.append(f"t={tr.t}: a={tr.action!s:<34} r={tr.reward:+6.1f}  -> s{tr.t + 1} = {tr.next_state}")
        return "\n".join(lines)


# ---------------------------------------------------------------------------
# 2. Discounted return
# ---------------------------------------------------------------------------
def discounted_return(rewards: Sequence[float], gamma: float = GAMMA) -> float:
    """G0 = sum_{t=0}^{T} gamma^t * r_t"""
    if not 0.0 <= gamma <= 1.0:
        raise ValueError("gamma must be in [0, 1]")
    return float(sum((gamma ** t) * r for t, r in enumerate(rewards)))


# ---------------------------------------------------------------------------
# 3. Empirical transition frequency matrix  P(s' | s, a)
# ---------------------------------------------------------------------------
# Historical jump records through the Homunculus Nebula: (sector, action, resulting sector).
HISTORICAL_JUMPS: List[Tuple[str, str, str]] = (
    [("outer_shell", "warp_escape_corridor", "escape_corridor")] * 14
    + [("outer_shell", "warp_escape_corridor", "dust_lane")] * 4
    + [("outer_shell", "warp_escape_corridor", "outer_shell")] * 2
    + [("outer_shell", "warp_direct", "dust_lane")] * 9
    + [("outer_shell", "warp_direct", "homunculus_core")] * 6
    + [("outer_shell", "warp_direct", "escape_corridor")] * 5
    + [("dust_lane", "warp_escape_corridor", "escape_corridor")] * 11
    + [("dust_lane", "warp_escape_corridor", "dust_lane")] * 5
    + [("dust_lane", "warp_direct", "homunculus_core")] * 8
    + [("dust_lane", "warp_direct", "dust_lane")] * 4
    + [("homunculus_core", "warp_escape_corridor", "dust_lane")] * 7
    + [("homunculus_core", "warp_escape_corridor", "homunculus_core")] * 6
    + [("homunculus_core", "warp_escape_corridor", "escape_corridor")] * 2
    + [("homunculus_core", "warp_direct", "homunculus_core")] * 10
    + [("homunculus_core", "warp_direct", "dust_lane")] * 2
)


def transition_frequency_matrix(
    dataset: Iterable[Tuple[Hashable, Hashable, Hashable]],
) -> Dict[Tuple[Hashable, Hashable], Dict[Hashable, float]]:
    """Maximum-likelihood estimate P(s'|s,a) = N(s,a,s') / N(s,a)."""
    counts: Dict[Tuple[Hashable, Hashable], Counter] = defaultdict(Counter)
    for s, a, s_next in dataset:
        counts[(s, a)][s_next] += 1
    probs = {}
    for sa, c in counts.items():
        total = sum(c.values())
        probs[sa] = {s_next: n / total for s_next, n in sorted(c.items())}
    return probs


def as_table(P, states: Optional[List[Hashable]] = None) -> Tuple[List[Tuple], List[Hashable], List[List[float]]]:
    """Return (row_labels, column_labels, matrix) — one row per (s, a)."""
    if states is None:
        states = sorted({s2 for row in P.values() for s2 in row} | {s for s, _ in P})
    rows = sorted(P)
    matrix = [[P[sa].get(s2, 0.0) for s2 in states] for sa in rows]
    return rows, states, matrix


def print_matrix(P, states=None) -> None:
    rows, cols, M = as_table(P, states)
    w = max(len(f"{s} | {a}") for s, a in rows) + 2
    print("P(s' | s, a)".ljust(w) + "".join(c[:15].rjust(17) for c in cols))
    for (s, a), r in zip(rows, M):
        print(f"{s} | {a}".ljust(w) + "".join(f"{p:17.3f}" for p in r))


In [4]:
P = transition_frequency_matrix(HISTORICAL_JUMPS)
print(f"{len(HISTORICAL_JUMPS)} historical jumps")
print_matrix(P)

95 historical jumps
P(s' | s, a)                                    dust_lane  escape_corridor  homunculus_core      outer_shell
dust_lane | warp_direct                             0.333            0.000            0.667            0.000
dust_lane | warp_escape_corridor                    0.312            0.688            0.000            0.000
homunculus_core | warp_direct                       0.167            0.000            0.833            0.000
homunculus_core | warp_escape_corridor              0.467            0.133            0.400            0.000
outer_shell | warp_direct                           0.450            0.250            0.300            0.000
outer_shell | warp_escape_corridor                  0.200            0.700            0.000            0.100


## Part 4 — Security & Ethics Sentinel: Cycle Detection & Halting Guardrail
**Owner: Arjun.** Replace this cell's code with Part 4: Floyd's Tortoise and Hare, O(1) state-action hashing, and the Emergency Tripwire that calls `agent.halt(...)`.

In [5]:
# TODO (Arjun): Part 4 code goes here.
# Required names used by Test 4 below:
#   state_action_hash(state, action) -> str
#   floyd_cycle_detect(sequence) -> (has_cycle: bool, mu: int, lam: int)
#   class CycleSentinel: observe(agent, state, action) -> bool  (True = tripwire fired, agent HALTED)


## Integration demo (Parts 1–3)
AURA flies out of the nebula. Its first shield command is wrong (−20 %); the error boundary returns it as an observation and the next deliberation fixes it. Every step is logged as an MDP transition.

In [6]:
PLAN = [
    {"tool": "query_sensor_array",   "arguments": {"target": "escape_corridor"}},
    {"tool": "adjust_plasma_shields", "arguments": {"intensity": -20}},     # System 1 mistake
    {"tool": "navigate_warp_vector", "arguments": {"x": -0.85, "y": 0.52, "z": 0.05}},
]
dispatcher, logger = ActionDispatcher(), TrajectoryLogger()
mission = {"i": 0, "sector": "outer_shell", "shields": 0, "pending": None, "obs": None}

def on_perceive(agent):
    logger.start(mission["sector"])

def on_deliberate(agent):
    last = agent.context[-1] if agent.context else None
    if last and last["is_error"] and "Shield" in last["content"]:
        mission["pending"] = {"tool": "adjust_plasma_shields", "arguments": {"intensity": 80}}  # self-correct
    else:
        mission["pending"] = PLAN[min(mission["i"], len(PLAN) - 1)]
        mission["i"] += 1

def on_act(agent):
    mission["obs"] = dispatcher.dispatch_into(agent, json.dumps(mission["pending"]))

def on_observe(agent):
    obs, call = mission["obs"], mission["pending"]
    reward = -5.0 if obs.startswith(ERROR_PREFIX) else 0.0
    if call["tool"] == "adjust_plasma_shields" and not obs.startswith(ERROR_PREFIX):
        mission["shields"] = call["arguments"]["intensity"]
    if call["tool"] == "navigate_warp_vector" and not obs.startswith(ERROR_PREFIX):
        mission["sector"] = "escape_corridor" if mission["shields"] >= 50 else "homunculus_core"
        reward = 20.0 if mission["sector"] == "escape_corridor" else -20.0
    logger.log(f"{call['tool']}({call['arguments']})", reward, mission["sector"],
               done=mission["sector"] == "escape_corridor")
    print(f"  {call['tool']:<22} -> {obs}")

def on_reflect(agent):
    agent.goal_reached = mission["sector"] == "escape_corridor"

aura = Agent(hooks={AgentState.PERCEIVING: on_perceive, AgentState.DELIBERATING: on_deliberate,
                    AgentState.ACTING: on_act, AgentState.OBSERVING: on_observe,
                    AgentState.REFLECTING: on_reflect})
aura.run()
print(f"\nFinal state: {aura.state.name} after {aura.step_count} steps ({aura.halt_reason})")
print(logger.pretty())
print(f"G0 (gamma=0.95) = {logger.discounted_return():.4f}")

  query_sensor_array     -> Observation: Clear gravitational corridor along vector [-0.85, 0.52, 0.05].
  adjust_plasma_shields  -> Tool Execution Error: ValueError: Shield intensity -20% out of range [0, 100].
  adjust_plasma_shields  -> Observation: Plasma shields set to 80%.
  navigate_warp_vector   -> Observation: Warp vector aligned to [-0.85, 0.52, 0.05].

Final state: HALTED after 18 steps (goal reached)
s0 = outer_shell
t=0: a=query_sensor_array({'target': 'escape_corridor'}) r=  +0.0  -> s1 = outer_shell
t=1: a=adjust_plasma_shields({'intensity': -20}) r=  -5.0  -> s2 = outer_shell
t=2: a=adjust_plasma_shields({'intensity': 80}) r=  +0.0  -> s3 = outer_shell
t=3: a=navigate_warp_vector({'x': -0.85, 'y': 0.52, 'z': 0.05}) r= +20.0  -> s4 = escape_corridor
G0 (gamma=0.95) = 12.3975


## Master Test Suite

In [7]:
# Test 1 — State Machine: IDLE ... -> HALTED, and strict termination when forced to run 21 steps
a = Agent()
assert a.state is AgentState.IDLE
expected = [AgentState.PERCEIVING, AgentState.DELIBERATING, AgentState.ACTING,
            AgentState.OBSERVING, AgentState.REFLECTING, AgentState.DELIBERATING]
assert [a.step() for _ in range(6)] == expected
a.goal_reached = True
for _ in range(4):
    a.step()
assert a.state is AgentState.HALTED and a.halt_reason == "goal reached"
for s, frm, to in a.history:
    assert to in TRANSITIONS[frm], (frm, to)

b = Agent(max_steps=20)
for _ in range(21):
    b.step()
assert b.state is AgentState.HALTED
assert b.step_count == 20 and "budget" in b.halt_reason
b.step(); assert b.state is AgentState.HALTED and b.step_count == 20     # stays halted
try:
    Agent(max_steps=21); raise AssertionError("budget > 20 must be rejected")
except ValueError:
    pass
print("Test 1 passed: state machine + step budget")

Test 1 passed: state machine + step budget


In [8]:
# Test 2 — Protocol Dispatch: invalid JSON -> formatted error string, no crash
d = ActionDispatcher()
bad_payloads = ['{"tool": "navigate_warp_vector", "arguments": {"x": 1,', 'not json at all', '[]', '{}',
                '{"tool": "adjust_plasma_shields", "arguments": {"intensity": "max"}}',
                '{"tool": "adjust_plasma_shields", "arguments": {"intensity": true}}',
                '{"tool": "navigate_warp_vector", "arguments": {"x": 1, "y": 2}}',
                '{"tool": "warp_core_eject", "arguments": {}}']
for p in bad_payloads:
    out = d.dispatch(p)
    assert isinstance(out, str) and out.startswith("Tool Execution Error: "), (p, out)

out = d.dispatch('{"tool": "adjust_plasma_shields", "arguments": {"intensity": -10}}')
assert out.startswith("Tool Execution Error: ValueError"), out             # tool exception caught
agent = Agent()
d.dispatch_into(agent, '{"tool": "adjust_plasma_shields", "arguments": {"intensity": -10}}')
assert agent.context[-1]["is_error"] and agent.context[-1]["content"].startswith("Tool Execution Error")
assert d.dispatch('{"tool": "adjust_plasma_shields", "arguments": {"intensity": 70}}').startswith("Observation:")
print("Test 2 passed: dispatcher validates and never crashes")

Test 2 passed: dispatcher validates and never crashes


In [9]:
# Test 3 — MDP Return: rewards [10, 0, -5, 20], gamma = 0.95
expected_G0 = 10 + 0.95 * 0 + 0.95**2 * (-5) + 0.95**3 * 20      # = 22.635
assert abs(discounted_return([10, 0, -5, 20], 0.95) - expected_G0) < 1e-9
assert abs(discounted_return([10, 0, -5, 20], 0.95) - 22.635) < 1e-9

lg = TrajectoryLogger(); lg.start("s0")
for i, r in enumerate([10, 0, -5, 20]):
    lg.log(f"a{i}", r, f"s{i+1}")
assert lg.as_tau() == ("s0", "a0", 10.0, "s1", "a1", 0.0, "s2", "a2", -5.0, "s3", "a3", 20.0, "s4")
assert abs(lg.discounted_return() - 22.635) < 1e-9
assert abs(lg.returns_to_go()[0] - 22.635) < 1e-9

P = transition_frequency_matrix(HISTORICAL_JUMPS)
for sa, row in P.items():
    assert abs(sum(row.values()) - 1.0) < 1e-12, sa
assert abs(P[("outer_shell", "warp_escape_corridor")]["escape_corridor"] - 14 / 20) < 1e-12
print(f"Test 3 passed: G0 = {discounted_return([10, 0, -5, 20]):.3f}")

Test 3 passed: G0 = 22.635


In [10]:
# Test 4 — Sentinel Cycle: repeating [s1, s2, s3, s1, s2, s3, ...] must halt the agent before step 10
# TODO (Arjun): write this test with assert statements once Part 4 is in.
